1.) What challenges did you encounter when loading and inspecting a raw external dataset from Kaggle compared to pre-packaged datasets, specifically regarding implicit data types like whitespace entries in TotalCharges?

Loading the raw dataset from Kaggle was straightforward. I used the same approach from HW2, pushing the CSV to GitHub and reading it in via the raw file URL, and pd.read_csv() handled it without issues. A challenge came during inspection. I will go into this in more detail in question 2. 

2.) How did you determine which columns required missing value imputation versus other data cleaning methods, and what strategy did you choose?

TotalCharges looked like it should be a numeric column of dollar amounts, but df.info() showed it was stored as object dtype instead of float64. This turned out to be a case of implicit missing data. df.isnull().sum() reported zero nulls for the column while the column actually contained whitespace strings that were being treated as valid entries. I had to check for this manually by stripping whitespace and comparing to an empty string, which revealed 11 rows with blank TotalCharges values, all of which had tenure equal to 0, meaning these were customers who hadn't been billed yet rather than customers with unknown totals. Once identified, I converted the column using pd.to_numeric(TotalCharges, errors='coerce'), which turned those whitespace strings into proper NaN values that SimpleImputer could then detect and impute with 0.



3.) Explain how the Scikit-Learn ColumnTransformer and Pipeline streamline the data preparation process and prevent data leakage across preprocessing and training steps.

Scikit-Learn's Pipeline combines multiple preprocessing steps into one process, while ColumnTransformer lets you apply different preprocessing to different columns. They also help prevent data leakage by making sure preprocessing is fit only on the training data. This keeps the test data unseen and gives a more accurate evaluation of the model.

4.) How did comparing StandardScaler versus MinMaxScaler impact your model training, convergence, or final evaluation metrics?

For this assignment, I used StandardScaler instead of testing both scalers. StandardScaler scales features based on their mean and standard deviation, while MinMaxScaler scales them between 0 and 1. Since SGDClassifier is affected by feature scale and features like TotalCharges have high outliers, StandardScaler should help the model converge more consistently. I did not directly compare the final evaluation metrics between the two scalers.

(Note: I’m a bit confused by this. Was I supposed to use both? In the task, it says “(StandardScaler or MinMaxScaler).” Did I misinterpret this?)



5.) How does configuring SGDClassifier(loss='log_loss') differ from standard linear classification models, and what advantages do probability outputs and ROC-AUC metrics provide when evaluating customer churn?

Using loss='log_loss' makes SGDClassifier perform logistic regression and provides probability estimates through predict_proba(). This is useful for churn prediction because the model can rank customers by their likelihood of churning and adjust the prediction threshold based on business needs. ROC-AUC measures how well the model separates churners from non-churners across different thresholds rather than relying on just one threshold.

6.) Based on your model coefficients or feature evaluation, which features show the strongest relationship with customer churn, and what operational business insights do these findings reveal?

Based on the feature coefficients, tenure was the strongest predictor of churn, with a coefficient of -1.31. This suggests that customers with longer tenure are less likely to churn. Month-to-month contracts (0.95) and fiber optic internet (0.94) were the next strongest positive predictors, meaning these customers were more likely to churn. TotalCharges (0.85), PaperlessBilling (0.68), and StreamingTV (0.59) were also important predictors. These results suggest that newer customers may be at higher risk of leaving, so retention efforts could focus more on them. Month-to-month customers may also be more likely to leave, making longer-term contracts a possible area to focus on. The strong relationship between fiber optic service and churn is also worth investigating to understand why these customers may be leaving.